# Step 3 · Download every city into SQLite

**Goal:** save in `data/climate.db` the daily temperature (mean, max and min) from 1940 to the
last complete year for the 140 cities of `data/cities.csv`.

From the previous steps:
- each city costs **~673 calls**;
- limits: 600/minute, 5,000/hour, 10,000/day. We use a **budget of 9,000 a day**, so
  **13 cities per batch**: about 11 days for the 140.

**How it works** (`pipelines.download`):
1. Before each city the batch checks the **budget** (the calls of the last 24 h, recorded by us
   in the `usage` table) and waits until **9 minutes** have passed since the last request.
2. The city is downloaded, **validated** and saved **in one transaction**: either its 31,412
   days are saved and it becomes `complete`, or nothing is saved.
3. **Resumable:** running it again only requests the cities that are not `complete`. Complete
   cities never spend another call.

**Use:** run the whole notebook once a day. It can be stopped at any time (Jupyter's ■ button,
or a power cut): what is complete stays saved. A city that fails is retried in the next batches,
up to 3 times.

**Already have the data?** No need to wait 11 days. Section 4 imports, in this order:
1. **`data/climate.parquet`**, the shared copy of the data (~23 MB). Ask a team member for it and
   put it in `data/`; whoever has the complete database can create it in section 7.
2. The database of the original *clima-mundial* project, if it is next to this repository
   (`../clima-mundial/data/clima.db`).

## 1. Configuration

In [1]:
from datetime import date
from pathlib import Path

import pandas as pd
from pipelines.download import (
    DownloadConfig,
    days_between,
    export_parquet,
    file_logger,
    import_clima_mundial,
    import_parquet,
    load_cities,
    open_db,
    read_cities_csv,
    run_batch,
    usage_24h,
)
from pipelines.open_meteo import request_cost

# Project root: Jupyter (and nbconvert) start in the notebook folder (notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CITIES_CSV = ROOT / "data" / "cities.csv"
DB_PATH = ROOT / "data" / "climate.db"
TEST_DB_PATH = ROOT / "data" / "test.db"
LOG_PATH = ROOT / "data" / "download.log"  # follow the progress with `tail -f`

START = "1940-01-01"
END = f"{date.today().year - 1}-12-31"  # last complete year

CONFIG = DownloadConfig(daily_budget=9_000, pause_minutes=9)

# Existing data, imported in section 4 if the files exist
SHARED_PARQUET = ROOT / "data" / "climate.parquet"  # shared copy (see section 7)
CLIMA_MUNDIAL_DB = ROOT.parent / "clima-mundial" / "data" / "clima.db"

RUN_TEST = True  # cheap test (1 call) in data/test.db
RUN_BATCH = True  # real download into data/climate.db
EXPORT_PARQUET = (
    True  # section 7: write data/climate.parquet when every city is complete
)

log = file_logger(LOG_PATH)
city_cost = request_cost(days_between(START, END))
print(
    f"Period: {START} → {END} · {days_between(START, END):,} days per city "
    f"→ {city_cost:.1f} calls"
)
print(f"Cities per batch: {int(CONFIG.daily_budget // city_cost)}")

Period: 1940-01-01 → 2025-12-31 · 31,412 days per city → 673.1 calls
Cities per batch: 13


## 2. The database

Four tables (`SCHEMA` in `pipelines.download`):

| Table | One row per… | What for |
|---|---|---|
| `cities` | city | copy of `cities.csv` + the ERA5 cell returned by the API (`era5_lat`, `era5_lon`, `era5_elevation`) |
| `downloads` | city | status (`pending` / `complete` / `error`), attempts, last error, warnings |
| `daily` | city and day | `t_mean`, `t_max`, `t_min`. Primary key `(city_id, date)`: a day cannot be duplicated |
| `usage` | API request | time, city, calls spent, HTTP status. It is our quota counter |

`CREATE TABLE IF NOT EXISTS` lets the notebook run many times without deleting anything, and
SQLite's **WAL** mode lets other programs read the database while the batch writes.

What the batch does with each API response (`fetch_with_retries`):

| Response | What happens |
|---|---|
| `200` | record the cost and return the data |
| `200` **empty** or unreadable | wait 65 s and retry (the cost is recorded: the API charges it) |
| `429` per **minute** | wait 65 s and retry |
| `429` per **hour** | wait 1 hour and retry |
| `429` **daily** | **stop the batch**; completed cities stay saved |
| `5xx` or network error | retry after 30 s, 60 s, 120 s |
| other `4xx` | request error: retrying makes no sense |

The empty `200` happened 3 times in the original project (Cuiabá, Vila Velha, Yichun): the server
takes ~30 s and sends a `200` with no data.

## 3. Cheap test: 1 call

Before spending real quota, the whole circuit is tested in a separate database
(`data/test.db`, deleted and created again each time):
- **one city** (the first of the CSV) and **one month** (January 2024) → cost: **1 call**;
- no pause.

Then the batch runs **a second time**: the city is already `complete`, so it must not make any
request. That checks that the process is resumable.

In [2]:
if RUN_TEST:
    for suffix in ("", "-wal", "-shm"):
        Path(f"{TEST_DB_PATH}{suffix}").unlink(missing_ok=True)

    test_db = open_db(TEST_DB_PATH)
    load_cities(test_db, read_cities_csv(CITIES_CSV).head(1))
    test_config = DownloadConfig(pause_minutes=0)

    run_batch(test_db, "2024-01-01", "2024-01-31", test_config, log, db_name="test.db")
    first = test_db.execute("SELECT COUNT(*), SUM(calls) FROM usage").fetchone()
    run_batch(test_db, "2024-01-01", "2024-01-31", test_config, log, db_name="test.db")
    second = test_db.execute("SELECT COUNT(*), SUM(calls) FROM usage").fetchone()

    print(f"\nRequests after the 1st run: {first[0]} ({first[1]} calls)")
    print(f"Requests after the 2nd run: {second[0]} ({second[1]} calls)")
    assert first == (1, 1.0), "The test should cost exactly 1 call"
    assert second == first, "The 2nd run should not make any request"
    print("OK: 1 call, and the second run repeats nothing.")
    display(
        pd.read_sql(
            "SELECT id, name, latitude, longitude, era5_lat, era5_lon, era5_elevation FROM cities",
            test_db,
        )
    )
    display(pd.read_sql("SELECT * FROM daily LIMIT 5", test_db))
    test_db.close()

2026-11-02 15:53:42  === Batch in test.db: 1 pending cities ===


2026-11-02 15:53:42  [1/1] Phoenix (phoenix-us) · usage 24 h: 0


2026-11-02 15:53:44    ✓ 31 days saved (cell 33.5, -112.0)


2026-11-02 15:53:44  === End of batch: 1 cities completed ===


2026-11-02 15:53:44  === Batch in test.db: 0 pending cities ===


2026-11-02 15:53:44  No pending city left.



Requests after the 1st run: 1 (1.0 calls)
Requests after the 2nd run: 1 (1.0 calls)
OK: 1 call, and the second run repeats nothing.


,id,name,latitude,longitude,era5_lat,era5_lon,era5_elevation
0,phoenix-us,Phoenix,33.5419,-112.0719,33.5,-112.0,366.0


,city_id,date,t_mean,t_max,t_min
0,phoenix-us,2024-01-01,10.6,16.5,6.2
1,phoenix-us,2024-01-02,10.7,17.5,4.5
2,phoenix-us,2024-01-03,10.4,16.8,4.4
3,phoenix-us,2024-01-04,10.0,15.5,5.1
4,phoenix-us,2024-01-05,9.1,13.8,4.0


## 4. The real database: create it, load the cities, import existing data

`load_cities` runs on every batch: it creates the tables if needed and updates the cities from
the CSV. New cities start as `pending`; downloaded ones are not touched. Cities that left the CSV
(for example after changing `FORCE` in notebook 02) are deleted **only if they have no data**.

Then the existing data, if there is any:
- `import_parquet` reads the shared copy `data/climate.parquet` (written by section 7) and
  **validates every city like a download** (every day of the period, no empty values,
  `t_min ≤ t_mean ≤ t_max`) before saving it.
- `import_clima_mundial` copies the cities already downloaded by the *clima-mundial* project
  (same ids, same data).

Both only take the cities of our CSV that are not complete yet, so they are safe to run again.
Whatever is still missing is downloaded by the batch of section 5.

In [3]:
connection = open_db(DB_PATH)
removed = load_cities(connection, read_cities_csv(CITIES_CSV))
if removed:
    log(f"Pending cities no longer in the CSV, deleted: {', '.join(removed)}")

if SHARED_PARQUET.exists():
    log(
        f"Imported from {SHARED_PARQUET.name}: {import_parquet(connection, SHARED_PARQUET)} cities"
    )
if CLIMA_MUNDIAL_DB.exists():
    log(
        f"Imported from clima-mundial: {import_clima_mundial(connection, CLIMA_MUNDIAL_DB)} cities"
    )
if not SHARED_PARQUET.exists() and not CLIMA_MUNDIAL_DB.exists():
    print("No existing data: the cities will be downloaded from the API (section 5).")


def show_status(connection):
    status = pd.read_sql(
        """SELECT c.priority, d.status, COUNT(*) AS cities
           FROM downloads d JOIN cities c ON c.id = d.city_id
           GROUP BY c.priority, d.status""",
        connection,
    )
    display(
        status.pivot(index="priority", columns="status", values="cities")
        .fillna(0)
        .astype(int)
    )
    print(
        f"Usage in the last 24 h: {usage_24h(connection):,.0f} "
        f"of {CONFIG.daily_budget:,.0f} calls"
    )


show_status(connection)

2026-11-02 15:53:44  Imported from clima-mundial: 0 cities


status,complete
priority,
1,28
2,56
3,56


Usage in the last 24 h: 0 of 9,000 calls


## 5. Today's batch

⚠️ This cell **takes time**: up to 13 cities with 9 minutes between them ≈ **2 hours**.
It can be stopped with ■ at any time without losing what is complete.

On Windows with WSL, if the computer goes to sleep the batch freezes and goes on when it wakes
up: nothing is lost, only time.

In [4]:
if RUN_BATCH:
    run_batch(connection, START, END, CONFIG, log, db_name=DB_PATH.name)
else:
    print("RUN_BATCH = False: nothing is downloaded.")

2026-11-02 15:53:44  === Batch in climate.db: 0 pending cities ===


2026-11-02 15:53:44  No pending city left.


## 6. Summary

In [5]:
show_status(connection)

print("\nCities with an error or a warning:")
display(
    pd.read_sql(
        """SELECT city_id, status, attempts, last_error, warning
       FROM downloads WHERE status = 'error' OR warning IS NOT NULL""",
        connection,
    )
)

rows = connection.execute("SELECT COUNT(*) FROM daily").fetchone()[0]
print(f"Rows in the daily table: {rows:,}")
print(f"Size of {DB_PATH.name}: {DB_PATH.stat().st_size / 1e6:.1f} MB")

status,complete
priority,
1,28
2,56
3,56


Usage in the last 24 h: 0 of 9,000 calls

Cities with an error or a warning:


,city_id,status,attempts,last_error,warning


Rows in the daily table: 4,397,680
Size of climate.db: 253.5 MB


Example SQL query: annual mean temperature of the first complete city.
`substr(date, 1, 4)` takes the first 4 characters of the date, that is, the year.

In [6]:
first_city = connection.execute(
    """SELECT c.id, c.name FROM cities c JOIN downloads d ON d.city_id = c.id
       WHERE d.status = 'complete' ORDER BY c.download_order LIMIT 1"""
).fetchone()

if first_city:
    annual = pd.read_sql(
        """SELECT substr(date, 1, 4) AS year, ROUND(AVG(t_mean), 2) AS annual_t_mean
           FROM daily WHERE city_id = ? GROUP BY year ORDER BY year""",
        connection,
        params=(first_city[0],),
    )
    print(f"{first_city[1]}: {len(annual)} years")
    display(annual.tail())
else:
    print("No complete city yet.")

Phoenix: 86 years


,year,annual_t_mean
81,2021,24.02
82,2022,23.77
83,2023,23.98
84,2024,24.27
85,2025,24.35


## 7. Share the data

Downloading the 140 cities takes about 11 days. Once the database is complete, `export_parquet`
writes it to **`data/climate.parquet`**: one row per city and day, with the ERA5 cell of each
city, compressed to ~23 MB (the database takes ~250 MB). Share that file (for example on Google
Drive) and anyone can import it in section 4 instead of downloading.

It is not in git: `data/` is ignored except `cities.csv`, and git is not good at storing large
binary files that change.

In [7]:
pending = connection.execute(
    "SELECT COUNT(*) FROM downloads WHERE status != 'complete'"
).fetchone()[0]
if EXPORT_PARQUET and pending == 0:
    n_cities = export_parquet(connection, SHARED_PARQUET)
    size = SHARED_PARQUET.stat().st_size / 1e6
    print(f"Written {SHARED_PARQUET.name}: {n_cities} cities, {size:.1f} MB")
elif EXPORT_PARQUET:
    print(f"{pending} cities are not complete yet: nothing exported.")

Written climate.parquet: 140 cities, 24.6 MB


In [8]:
connection.close()

## Conclusions

- **The download works end to end:** the 1-call test downloads Phoenix (January 2024, 31 days,
  ERA5 cell 33.5, −112.0) and the second run makes no request, so the batch is **resumable**.
- **The 140 cities are complete**, imported from the *clima-mundial* project: 4,397,680 days
  (140 × 31,412, from 1940-01-01 to 2025-12-31), with no errors or warnings. `climate.db` takes
  about 250 MB and is not in git (`data/` is ignored except `cities.csv`).
- **Shared copy:** section 7 writes `data/climate.parquet` (140 cities, ~25 MB). Imported in a
  fresh clone of the repository, it gives exactly the same results in notebooks 04 and 05,
  so nobody else has to wait for the download.
- **Without any existing data**, the same notebook downloads everything from the API:
  13 cities a day with a 9,000-call budget, so about 11 days. Real-life lessons from that
  download in the original project:
  - the budget is a **rolling 24-hour window**: launching a batch less than 24 h after the
    previous one only downloads the cities that fit, and the batch ends instead of waiting;
  - if the computer **goes to sleep**, the batch freezes and resumes when it wakes up;
  - the API sometimes answers with an **empty 200**; it is now retried on the spot.
- **Updating with a new year** costs only ~8 calls per city: it will need a function that
  requests the missing days and adds them (not done yet).